<a href="https://colab.research.google.com/github/ameesha543/Statistical-Learning-e23095/blob/main/Assignment_7/Assignment7b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


In [1]:
import numpy as np
import plotly.graph_objects as go


def item_probability(theta, a, b):
    """2PL probability of a correct response."""
    z = np.clip(a * (theta - b), -700, 700)
    return 1.0 / (1.0 + np.exp(-z))


theta = np.linspace(-4, 4, 500)

parameters = [
    (0.5, 0.0, "a = 0.5, b = 0"),
    (2.0, -1.0, "a = 2.0, b = -1"),
    (2.0, 0.0, "a = 2.0, b = 0"),
    (2.0, 1.0, "a = 2.0, b = 1")
]

fig = go.Figure()

for a, b, label in parameters:
    fig.add_trace(
        go.Scatter(
            x=theta,
            y=item_probability(theta, a, b),
            mode="lines",
            name=label
        )
    )

fig.update_layout(
    title="Two-Parameter Logistic Item Response Curves",
    xaxis_title="Ability, θ",
    yaxis_title="P(Yᵢ = 1 | Θ = θ)",
    yaxis=dict(range=[0, 1]),
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

## 2. Sequential likelihood contribution

For a single new response $y_k \in \{0, 1\}$, the likelihood contribution is the Bernoulli likelihood

$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$

where

$$p_k(\theta) = \frac{1}{1 + \exp[-a_k(\theta - b_k)]}.$$

Therefore,

$$L(y_k \mid \theta) =
\begin{cases}
p_k(\theta), & y_k = 1, \\
1 - p_k(\theta), & y_k = 0.
\end{cases}$$

Assuming that item responses are conditionally independent given $\Theta = \theta$, the joint likelihood for the running response vector

$$y^{(k)} = (y_1, \dots, y_k)$$

is

$$L(y^{(k)} \mid \theta) = \prod_{i=1}^k [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$
# 3. Mathematical formulation of the running update

Let  
$$f_{k-1}(\theta) = f_{\Theta|Y^{(k-1)}} \left( \theta \mid y^{(k-1)} \right)$$
denote the posterior after the first $k - 1$ responses.

After observing the new response $y_k$, the updated posterior is  
$$f_k(\theta) \propto \left[ p_k(\theta) \right]^{y_k} \left[ 1 - p_k(\theta) \right]^{1 - y_k} f_{k-1}(\theta)$$
or, using the single-response likelihood,  
$$f_k(\theta) \propto L(y_k \mid \theta) f_{k-1}(\theta)$$

The normalized posterior is  
$$f_k(\theta) = \frac{L(y_k \mid \theta) f_{k-1}(\theta)}{\int_{-\infty}^{\infty} L(y_k \mid u) f_{k-1}(u) \, du}$$

The complete posterior after $k$ responses can also be written as  
$$f_k(\theta) \propto f_0(\theta) \prod_{i=1}^k \left[ p_i(\theta) \right]^{y_i} \left[ 1 - p_i(\theta) \right]^{1 - y_i}.$$
# 4. Dynamic shifting after a correct answer

For a correct answer, $y_k = 1$, the update becomes  
$$f_k(\theta) \propto p_k(\theta) f_{k-1}(\theta).$$

Because $p_k(\theta)$ is an increasing function of $\theta$:  
- low ability values receive small likelihood values;  
- high ability values receive larger likelihood values.  

For a highly difficult item, $b_k$ is large. If $\theta \ll b_k$, then  
$$p_k(\theta) \approx 0.$$

Therefore, a correct response is unlikely under low values of $\theta$. Multiplying the previous posterior by $p_k(\theta)$ greatly reduces its density at low ability values while retaining more density at high ability values.

The shift can also be shown using the derivative of the log-posterior. For $y_k = 1$,  
$$\log f_k(\theta) = \log f_{k-1}(\theta) + \log p_k(\theta) + C,$$

where $C$ does not depend on $\theta$. Since  
$$\frac{d}{d\theta} \log p_k(\theta) = a_k [1 - p_k(\theta)] > 0,$$

the updated log-posterior is increasing at the location of the previous mode. Consequently, the posterior peak generally shifts to the right, indicating a higher estimated ability.

Thus, correctly answering a difficult item provides strong evidence that the user's ability is higher than previously estimated.
# 5. Effect of discrimination on posterior sharpness

The log-likelihood contribution from item $k$ is  
$$\ell_k(\theta) = y_k \log p_k(\theta) + (1 - y_k) \log[1 - p_k(\theta)].$$

Its first derivative is  
$$\frac{d\ell_k}{d\theta} = a_k [y_k - p_k(\theta)].$$

Its second derivative is  
$$\frac{d^2\ell_k}{d\theta^2} = -a_k^2 p_k(\theta) [1 - p_k(\theta)].$$

Therefore, the information supplied by the item is  
$$I_k(\theta) = a_k^2 p_k(\theta) [1 - p_k(\theta)].$$

The information is largest when  
$$p_k(\theta) = 0.5,$$

which occurs at  
$$\theta = b_k.$$
The maximum item information is

$$I_{k,\text{max}} = \frac{a_k^2}{4}.$$

Very large $a_k$
When $a_k$ is large:

- the item response curve is steep;
- small changes in ability cause large changes in response probability;
- the item can strongly distinguish between nearby ability levels;
- the posterior generally becomes narrower and sharper;
- the posterior variance can decrease substantially.

However, this strong effect mainly occurs when $b_k$ is near the user's likely ability. If the item is much too easy or much too difficult, then $p_k(\theta)$ is close to 0 or 1, and the information may still be small.

### Very small $a_k$

When $a_k$ is small:

- the response curve is flat;
- response probability changes slowly with ability;
- the response provides little information about $\theta$;
- the posterior changes only slightly;
- the reduction in posterior variance is small.
# 6. Numerical implementation using a fixed grid

Choose a fixed grid  
$$\theta_1, \theta_2, \dots, \theta_m$$

over a suitable range, such as  
$$-5 \leq \theta \leq 5.$$

Let the grid spacing be  
$$\Delta \theta = \theta_{j+1} - \theta_j.$$

## Step 1: Initialize the prior

Calculate  
$$f_0(\theta_j) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta_j^2}{2}\right)$$

at every grid point.

Normalize it numerically:  
$$f_0(\theta_j) \leftarrow \frac{f_0(\theta_j)}{\sum_{r=1}^m f_0(\theta_r) \Delta \theta}.$$
# Step 2: Calculate the new likelihood

After observing $y_k$, calculate

$$p_{k,j} = \frac{1}{1 + \exp[-a_k(\theta_j - b_k)]}.$$

Then

$$L_{k,j} = p_{k,j}^{y_k}(1 - p_{k,j})^{1 - y_k}.$$

# Step 3: Obtain the unnormalized posterior

$$\tilde{f}_k(\theta_j) = L_{k,j} f_{k-1}(\theta_j).$$

# Step 4: Normalize the posterior

Calculate the numerical normalization constant

$$Z_k \approx \sum_{j=1}^m \tilde{f}_k(\theta_j) \Delta \theta.$$

Then

$$f_k(\theta_j) = \frac{\tilde{f}_k(\theta_j)}{Z_k}.$$
This ensures that  

$$\sum_{j=1}^m f_k(\theta_j) \Delta \theta \approx 1.$$
# Step 5: Calculate the estimators

The posterior mean is

$$\hat{\theta}_{\text{Bayes}}^{(k)} \approx \sum_{j=1}^m \theta_j f_k(\theta_j) \Delta \theta$$

The MAP estimate is the grid point with the largest posterior density:

$$\hat{\theta}_{\text{MAP}}^{(k)} = \theta_{\arg \max_j f_k(\theta_j)}$$

The posterior variance can be calculated as

$$\text{Var}(\Theta \mid \mathbf{y}^{(k)}) \approx \sum_{j=1}^m \left( \theta_j - \hat{\theta}_{\text{Bayes}}^{(k)} \right)^2 f_k(\theta_j) \Delta \theta.$$

In [2]:
import numpy as np
import plotly.graph_objects as go


def logistic(x):
    """Numerically stable logistic function."""
    x = np.clip(x, -700, 700)
    return 1.0 / (1.0 + np.exp(-x))


def calculate_mean(theta_grid, density):
    """Posterior mean calculated using numerical integration."""
    return np.trapezoid(theta_grid * density, theta_grid)


def calculate_sd(theta_grid, density, mean):
    """Posterior standard deviation."""
    variance = np.trapezoid(
        (theta_grid - mean) ** 2 * density,
        theta_grid
    )
    return np.sqrt(variance)


# Reproducible random-number generator
rng = np.random.default_rng(42)

# True hidden ability and number of items
theta_true = 0.75
n_items = 20

# Fixed grid for theta
theta_grid = np.linspace(-5, 5, 4001)

# Initial standard normal prior
posterior = (
    np.exp(-0.5 * theta_grid**2)
    / np.sqrt(2.0 * np.pi)
)

# Normalize the prior over the finite grid
posterior /= np.trapezoid(posterior, theta_grid)

# Generate the item parameters
a_items = rng.uniform(0.5, 2.0, n_items)
b_items = rng.normal(0.0, 1.0, n_items)

# Store estimates beginning at step 0
initial_mean = calculate_mean(theta_grid, posterior)
initial_map = theta_grid[np.argmax(posterior)]
initial_sd = calculate_sd(
    theta_grid,
    posterior,
    initial_mean
)

posterior_means = [initial_mean]
map_estimates = [initial_map]
posterior_sds = [initial_sd]

responses = []
true_probabilities = []

# Sequential item-response process
for k in range(n_items):

    a_k = a_items[k]
    b_k = b_items[k]

    # True probability of a correct response
    p_true = logistic(
        a_k * (theta_true - b_k)
    )

    true_probabilities.append(p_true)

    # Simulate y_k by comparing U(0,1) with p_true
    u_k = rng.uniform(0.0, 1.0)
    y_k = int(u_k < p_true)

    responses.append(y_k)

    # Probability of a correct answer over the theta grid
    p_grid = logistic(
        a_k * (theta_grid - b_k)
    )

    # Likelihood of the observed response
    likelihood = (
        p_grid**y_k
        * (1.0 - p_grid)**(1 - y_k)
    )

    # Unnormalized posterior
    unnormalized_posterior = posterior * likelihood

    # Sequential normalization
    normalization_constant = np.trapezoid(
        unnormalized_posterior,
        theta_grid
    )

    if normalization_constant <= 0:
        raise ValueError("Posterior normalization failed.")

    posterior = (
        unnormalized_posterior
        / normalization_constant
    )

    # Running posterior mean
    mean_k = calculate_mean(
        theta_grid,
        posterior
    )

    # Running MAP estimate
    map_k = theta_grid[np.argmax(posterior)]

    # Running posterior standard deviation
    sd_k = calculate_sd(
        theta_grid,
        posterior,
        mean_k
    )

    posterior_means.append(mean_k)
    map_estimates.append(map_k)
    posterior_sds.append(sd_k)


# Display the simulation results
print("Responses:", responses)

print(
    f"Final posterior mean: "
    f"{posterior_means[-1]:.4f}"
)

print(
    f"Final MAP estimate: "
    f"{map_estimates[-1]:.4f}"
)

print(
    f"Final posterior standard deviation: "
    f"{posterior_sds[-1]:.4f}"
)

print(
    f"True ability: {theta_true:.4f}"
)


# Plot estimator progression
steps = np.arange(n_items + 1)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode="lines+markers",
        name="Posterior Mean"
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP Estimate"
    )
)

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True ability = 0.75",
    annotation_position="top right"
)

fig.update_layout(
    title="Sequential Bayesian Estimation of User Ability",
    xaxis_title="Number of Items Answered, k",
    yaxis_title="Estimated Ability",
    xaxis=dict(
        tickmode="linear",
        tick0=0,
        dtick=1
    ),
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

Responses: [1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1]
Final posterior mean: 0.9536
Final MAP estimate: 0.9175
Final posterior standard deviation: 0.3908
True ability: 0.7500


# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

In [3]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta


# Avoid numerical issues exactly at the endpoints
theta = np.linspace(0.001, 0.999, 1000)

parameter_pairs = [
    (1, 1, "Beta(1, 1): Uniform"),
    (2, 8, "Beta(2, 8): Right-skewed"),
    (8, 2, "Beta(8, 2): Left-skewed")
]

fig = go.Figure()

for alpha, beta_parameter, label in parameter_pairs:
    density = beta.pdf(
        theta,
        a=alpha,
        b=beta_parameter
    )

    fig.add_trace(
        go.Scatter(
            x=theta,
            y=density,
            mode="lines",
            name=label
        )
    )

fig.update_layout(
    title="Probability Density Functions of Beta Distributions",
    xaxis_title="Click-through rate, θ",
    yaxis_title="Probability density",
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

# Interpretation

The mean of a Beta distribution is

$$E[\Theta] = \frac{\alpha}{\alpha + \beta}.$$

Beta(1, 1)

$$E[\Theta] = \frac{1}{1 + 1} = 0.5.$$

Its density is constant:

$$f(\theta) = 1.$$

Therefore, all CTR values between 0 and 1 are initially equally likely.

Beta(2, 8)

$$E[\Theta] = \frac{2}{2 + 8} = 0.2.$$

Most of the probability mass lies near low CTR values. It has a longer tail toward the right, so it is right-skewed.
Its mode is

$$\frac{\alpha - 1}{\alpha + \beta - 2} = \frac{2 - 1}{2 + 8 - 2} = 0.125.$$

Beta(8, 2)

$$E[\Theta] = \frac{8}{8 + 2} = 0.8.$$

Most of the probability mass lies near high CTR values. It has a longer tail toward the left, so it is left-skewed.

Its mode is

$$\frac{8 - 1}{8 + 2 - 2} = 0.875.$$

## General effect of $\alpha$ and $\beta$

- When $\alpha = \beta$, the distribution is symmetric around 0.5.
- When $\alpha < \beta$, the probability mass is concentrated toward $\theta = 0$.
- When $\alpha > \beta$, the probability mass is concentrated toward $\theta = 1$.
- Increasing both $\alpha$ and $\beta$ while keeping their ratio similar makes the distribution narrower and more concentrated.

The center of mass is determined by

$$\frac{\alpha}{\alpha + \beta}.$$
## 2. Sequential likelihood and joint history

### Likelihood of one response

For one isolated response $y_k$, the Bernoulli likelihood is

$$L(y_k | \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

because $y_k \in \{0, 1\}$.

For a click, $y_k = 1$,

$$L(1 | \theta) = \theta.$$

For a non-click, $y_k = 0$,

$$L(0 | \theta) = 1 - \theta.$$

### Joint likelihood of the running history

Let

$$y^{(k)} = (y_1, y_2, \dots, y_k).$$

Assuming conditional independence,

$$L(y^{(k)} | \theta) = \prod_{i=1}^k \theta^{y_i} (1 - \theta)^{1 - y_i}.$$
Therefore,

$$L(y^{(k)} | \theta) = \theta^{\sum_{i=1}^k y_i} (1 - \theta)^{k - \sum_{i=1}^k y_i}$$

Define the number of clicks as

$$S_k = \sum_{i=1}^k y_i$$

and the number of non-clicks as

$$F_k = k - S_k.$$

Then

$$L(y^{(k)} | \theta) = \theta^{S_k} (1 - \theta)^{F_k}$$
### 3. Closed-form analytical updates

Suppose that after step $k - 1$,

$$\Theta \mid \mathbf{Y}^{(k-1)} = \mathbf{y}^{(k-1)} \sim \text{Beta}(\alpha_{k-1}, \beta_{k-1}).$$

The previous posterior density is

$$f_{k-1}(\theta) = \frac{1}{B(\alpha_{k-1}, \beta_{k-1})} \theta^{\alpha_{k-1}-1}(1 - \theta)^{\beta_{k-1}-1}.$$

After observing the new response $y_k$, Bayes' theorem gives

$$f_k(\theta) \propto L(y_k \mid \theta) f_{k-1}(\theta).$$

Substituting the expressions,

$$f_k(\theta) \propto \theta^{y_k}(1 - \theta)^{1 - y_k} \theta^{\alpha_{k-1}-1}(1 - \theta)^{\beta_{k-1}-1}.$$

Combining the powers of $\theta$,

$$f_k(\theta) \propto \theta^{\alpha_{k-1} + y_k - 1}(1 - \theta)^{\beta_{k-1} + 1 - y_k - 1}.$$

This has the form of another Beta density. Therefore,

$$\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)} \sim \text{Beta}(\alpha_k, \beta_k).$$
where

$$\alpha_k = \alpha_{k-1} + y_k$$

and

$$\beta_k = \beta_{k-1} + 1 - y_k.$$

Thus:

- For $y_k = 1$,

$$\alpha_k = \alpha_{k-1} + 1, \quad \beta_k = \beta_{k-1}.$$

- For $y_k = 0$,

$$\alpha_k = \alpha_{k-1}, \quad \beta_k = \beta_{k-1} + 1.$$

This proves Beta-Bernoulli conjugacy.
# Update using all observations

After $k$ impressions,

$$\alpha_k = \alpha_0 + \sum_{i=1}^k y_i = \alpha_0 + S_k$$

and

$$\beta_k = \beta_0 + k - \sum_{i=1}^k y_i = \beta_0 + F_k.$$

Hence,

$$\Theta \mid \mathbf{y}^{(k)} \sim \text{Beta}(\alpha_0 + S_k, \beta_0 + F_k).$$

# Posterior density

The normalized posterior is

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{\theta^{\alpha_k - 1}(1 - \theta)^{\beta_k - 1}}{B(\alpha_k, \beta_k)}$$
# Posterior mean

The posterior mean at step $k$ is

$$E[\Theta \mid \mathbf{Y}^{(k)}] = \mathbf{y}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$$

Using the original parameters,

$$E[\Theta \mid \mathbf{y}^{(k)}] = \frac{\alpha_0 + S_k}{\alpha_0 + \beta_0 + k}.$$

For a uniform prior, $\alpha_0 = \beta_0 = 1$,

$$E[\Theta \mid \mathbf{y}^{(k)}] = \frac{S_k + 1}{k + 2}.$$
# 4. Dynamic shifting mechanics

## Effect of a click

When $y_k = 1$,

$$f_k(\theta) \propto \theta f_{k-1}(\theta).$$

Since $\theta$ is small near zero and large near one, multiplying by $\theta$:

- suppresses density near $\theta = 0$;
- retains more density near $\theta = 1$;
- shifts the posterior distribution toward higher CTR values.

The parameter update is

$$\alpha_k = \alpha_{k-1} + 1, \quad \beta_k = \beta_{k-1}.$$

The posterior mean changes from

$$\frac{\alpha_{k-1}}{\alpha_{k-1} + \beta_{k-1}}$$

to

$$\frac{\alpha_{k-1} + 1}{\alpha_{k-1} + \beta_{k-1} + 1},$$

which is larger than the previous mean.

Therefore, a click shifts the posterior toward the right.
# Effect of a non-click

When $y_k = 0$,

$$f_k(\theta) \propto (1 - \theta)f_{k-1}(\theta).$$

The factor $1 - \theta$:

- is large near $\theta = 0$;
- is small near $\theta = 1$;
- suppresses high CTR values;
- shifts the posterior toward lower CTR values.

The update is

$$\alpha_k = \alpha_{k-1}, \quad \beta_k = \beta_{k-1} + 1.$$

The new mean is

$$\frac{\alpha_{k-1}}{\alpha_{k-1} + \beta_{k-1} + 1},$$

which is lower than the previous mean.

Therefore, a non-click shifts the posterior toward the left.

# Effect on the posterior peak

When $\alpha_k > 1$ and $\beta_k > 1$, the posterior mode is

$$\theta_{\text{mode}} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}.$$
A click increases the numerator through $\alpha_k$, shifting the mode upward. A non-click increases $\beta_k$, increasing the denominator without increasing the numerator, shifting the mode downward.

## Comparison with the 2PL IRT model

For the Beta–Bernoulli model, the prior and posterior belong to the same Beta family. Therefore, the update only requires

$$\alpha_k = \alpha_{k-1} + y_k, \quad \beta_k = \beta_{k-1} + 1 - y_k.$$

No numerical integration is needed.

In the 2PL IRT model, the likelihood contains the logistic function

$$p_i(\theta) = \frac{1}{1 + e^{-a_i(\theta - b_i)}}.$$

A normal prior multiplied by this logistic likelihood does not produce a standard closed-form posterior distribution. Therefore, a numerical approximation is needed, such as:

- fixed-grid numerical integration;
- numerical quadrature;
- Laplace approximation;
- Markov chain Monte Carlo.

The Beta–Bernoulli model is therefore computationally simpler because of conjugacy.
# 5. Running point estimators

At step $k$,

$$\Theta \mid y^{(k)} \sim \text{Beta}(\alpha_k, \beta_k).$$

## Posterior mean estimator

$$\hat{\theta}_{Bayes}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$$

Using the initial prior and observed counts,

$$\hat{\theta}_{Bayes}^{(k)} = \frac{\alpha_0 + S_k}{\alpha_0 + \beta_0 + k}.$$

For $\alpha_0 = \beta_0 = 1$,

$$\hat{\theta}_{Bayes}^{(k)} = \frac{S_k + 1}{k + 2}.$$

This is also called the Bayes estimator under squared-error loss.
# MAP estimator

For

$$\alpha_k > 1, \quad \beta_k > 1,$$

the MAP estimator is

$$\hat{\theta}_{\text{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}.$$

For a uniform prior,

$$\alpha_k = 1 + S_k, \quad \beta_k = 1 + k - S_k.$$

Therefore, once both a click and a non-click have been observed,

$$\hat{\theta}_{\text{MAP}}^{(k)} = \frac{S_k}{k}.$$

Thus, with a uniform prior, the MAP estimate becomes the observed sample CTR.
# Boundary cases

The usual MAP formula requires $\alpha_k > 1$ and $\beta_k > 1$.

- If $\alpha_k = 1$ and $\beta_k > 1$, the mode is at  
  $$\hat{\theta}_{\text{MAP}}^{(k)} = 0.$$

- If $\alpha_k > 1$ and $\beta_k = 1$, the mode is at  
  $$\hat{\theta}_{\text{MAP}}^{(k)} = 1.$$

- If $\alpha_k = \beta_k = 1$, the distribution is uniform, so every point in $[0, 1]$ is a mode. For plotting, 0.5 may be used as a representative initial MAP value.

# Posterior variance

Although not directly requested, posterior certainty can be measured using  

$$\text{Var}(\Theta \mid \mathbf{y}^{(k)}) = \frac{\alpha_k \beta_k}{(\alpha_k + \beta_k)^2 (\alpha_k + \beta_k + 1)}.$$

As more impressions are observed, $\alpha_k + \beta_k$ increases and the posterior variance generally decreases.

In [4]:
import numpy as np
import plotly.graph_objects as go


def beta_mean(alpha, beta):
    """Posterior mean of a Beta distribution."""
    return alpha / (alpha + beta)


def beta_map(alpha, beta):
    """
    MAP estimate of a Beta distribution.

    For Beta(1,1), the mode is not unique.
    The midpoint 0.5 is used for visualization.
    """
    if alpha > 1 and beta > 1:
        return (alpha - 1) / (alpha + beta - 2)

    if alpha == 1 and beta > 1:
        return 0.0

    if alpha > 1 and beta == 1:
        return 1.0

    if alpha == 1 and beta == 1:
        return 0.5

    # General case when both parameters are below one:
    # the density has modes at both boundaries.
    return np.nan


def beta_variance(alpha, beta):
    """Posterior variance of a Beta distribution."""
    numerator = alpha * beta
    denominator = (
        (alpha + beta) ** 2
        * (alpha + beta + 1)
    )
    return numerator / denominator


# --------------------------------------------------------
# Simulation settings
# --------------------------------------------------------

rng = np.random.default_rng(42)

theta_true = 0.35
n_impressions = 100

alpha_0 = 1.0
beta_0 = 1.0

alpha = alpha_0
beta = beta_0

# --------------------------------------------------------
# Storage beginning at step 0
# --------------------------------------------------------

steps = [0]
responses = []

posterior_means = [
    beta_mean(alpha, beta)
]

map_estimates = [
    beta_map(alpha, beta)
]

posterior_variances = [
    beta_variance(alpha, beta)
]

alpha_history = [alpha]
beta_history = [beta]

# --------------------------------------------------------
# Sequential simulation and analytical updating
# --------------------------------------------------------

for k in range(1, n_impressions + 1):

    # Draw a random value from U(0,1)
    uniform_draw = rng.uniform(0.0, 1.0)

    # Generate response:
    # click if U < theta_true
    y_k = int(uniform_draw < theta_true)

    responses.append(y_k)

    # Closed-form Beta-Bernoulli update
    alpha = alpha + y_k
    beta = beta + (1 - y_k)

    # Calculate running estimators
    mean_k = beta_mean(alpha, beta)
    map_k = beta_map(alpha, beta)
    variance_k = beta_variance(alpha, beta)

    # Store results
    steps.append(k)
    posterior_means.append(mean_k)
    map_estimates.append(map_k)
    posterior_variances.append(variance_k)
    alpha_history.append(alpha)
    beta_history.append(beta)


# --------------------------------------------------------
# Final numerical information
# --------------------------------------------------------

number_of_clicks = sum(responses)
number_of_non_clicks = n_impressions - number_of_clicks

final_standard_deviation = np.sqrt(
    posterior_variances[-1]
)

print("Number of impressions:", n_impressions)
print("Number of clicks:", number_of_clicks)
print("Number of non-clicks:", number_of_non_clicks)

print(
    f"Observed sample CTR: "
    f"{number_of_clicks / n_impressions:.4f}"
)

print(
    f"Final posterior distribution: "
    f"Beta({alpha:.0f}, {beta:.0f})"
)

print(
    f"Final posterior mean: "
    f"{posterior_means[-1]:.4f}"
)

print(
    f"Final MAP estimate: "
    f"{map_estimates[-1]:.4f}"
)

print(
    f"Final posterior standard deviation: "
    f"{final_standard_deviation:.4f}"
)

print(
    f"True CTR: {theta_true:.4f}"
)


# --------------------------------------------------------
# Plot estimator progression
# --------------------------------------------------------

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode="lines",
        name="Posterior Mean"
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines",
        name="MAP Estimate"
    )
)

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True CTR = 0.35",
    annotation_position="top right"
)

fig.update_layout(
    title="Sequential Bayesian Tracking of Advertisement CTR",
    xaxis_title="Number of impressions, k",
    yaxis_title="Estimated click-through rate",
    xaxis=dict(
        range=[0, n_impressions],
        dtick=10
    ),
    yaxis=dict(
        range=[0, 1]
    ),
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

Number of impressions: 100
Number of clicks: 33
Number of non-clicks: 67
Observed sample CTR: 0.3300
Final posterior distribution: Beta(34, 68)
Final posterior mean: 0.3333
Final MAP estimate: 0.3300
Final posterior standard deviation: 0.0464
True CTR: 0.3500


# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In aerospace and civil engineering, Structural Health Monitoring (SHM) is critical for detecting damage before a catastrophic failure occurs. Consider an aircraft wing or a bridge girder equipped with specialized vibration sensors. Over time, environmental fatigue or dynamic impacts can cause micro-fractures, resulting in a reduction of the component's mechanical stiffness.

Let $\Theta = \theta$ represent the structural **remaining stiffness efficiency factor**, where $\theta$ is physically bounded to the interval:

$$\theta \in (0, 1]$$

* $\theta = 1.0$ indicates a perfectly pristine, undamaged structural component.
* $\theta \to 0$ signifies critical degradation or severe structural cracking.

Let $K_{\text{nominal}}$ be the known, baseline stiffness of the structural component when it is entirely healthy. At each sequential inspection time step $k$ (where $k = 1, 2, \dots, n$), a sensor collects a noisy experimental stiffness measurement $y_k$.

Engineers model the degradation physics via a non-linear relationship with multiplicative log-normal measurement noise to prevent non-physical negative values:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

where $\sigma$ is the standard deviation of the sensor noise in log-space.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running history vector of observed sensor readings** up to the current inspection milestone. Before deploying the sensors, engineers utilize an initial prior distribution $f_{\Theta}^{(0)}(\theta)$ over the domain $(0, 1]$ based on historical manufacturing specifications. As the sensor stream arrives, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Prior Belief Boundaries**

Before data collection begins, engineers assume the component is highly likely to be healthy, modeling this using a bounded Beta distribution as the initial prior: $\Theta \sim \text{Beta}(8, 1.5)$.

* Plot this initial prior density function using Plotly over the restricted physical domain $\theta \in [0.01, 1.0]$.
* Calculate the expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ analytically. Explain why this specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy.

#### **2. Structural Likelihood Formulation**

Using the change of variables or properties of the log-normal distribution, write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* continuous sensor measurement $y_k$ at inspection step $k$, given the true stiffness factor $\theta$. Following this, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

Explain why an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist when combining a Beta prior with this log-normal structural likelihood. Write down the recursive relationship for the posterior density at step $k$ up to a proportionality constant.

#### **4. Running Point Estimates**

Because a closed-form formula is unavailable, we must define point estimators through numerical integration. Write down the definite integral equations over the bounded domain $(0, 1]$ required to compute:

* The **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* The **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **5. Algorithmic Grid Approximation and Normalization**

Describe the step-by-step numerical procedure to maintain this distribution on a discrete grid of $\theta$-values. Explicitly state how you would handle the boundary limits computationally and how you would perform the sequential normalization step using the trapezoidal rule after a new sensor reading $y_k$ is observed.

#### **6. Performance Tracking and Degradation Convergence Analysis**

Suppose an impact occurs, and the true, hidden remaining stiffness drops to $\theta_{\text{true}} = 0.68$. Write a Python script using Plotly to simulate an engineered monitoring timeline across $n = 15$ continuous sensor measurements ($K_{\text{nominal}} = 50.0 \text{ kN/mm}$, $\sigma = 0.15$):

* **Simulate Sensor Stream:** Programmatically generate noisy sensor readings $y_k$ by drawing random values from the underlying log-normal physics model centered at $\theta_{\text{true}}$.
* **Track Estimators:** Loop sequentially through each step. At each step, update the unnormalized grid, normalize it via `np.trapezoid`, and compute both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize Curves & Timeline:** Generate two plots:
1. A plot showing the progression of the full posterior density curves at milestones $k \in \{0, 1, 2, 5, 10, 15\}$.
2. A line chart tracking the convergence of both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ from step $0$ to $15$ against a horizontal reference line at $\theta_{\text{true}} = 0.68$.


* **Analysis:** Evaluate the behavior of the distribution. How many sensor readings did it take for the system to overcome the initially optimistic "healthy" prior and confidently isolate the 68% damage state? What does the narrowing of the density curves imply about structural safety thresholds?

In [5]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta


# Restricted physical domain
theta_grid = np.linspace(0.01, 1.0, 2000)

alpha_0 = 8.0
beta_0 = 1.5

# Beta(8, 1.5) prior density
prior_density = beta.pdf(
    theta_grid,
    a=alpha_0,
    b=beta_0
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=theta_grid,
        y=prior_density,
        mode="lines",
        name="Beta(8, 1.5) prior"
    )
)

fig.update_layout(
    title="Initial Prior Distribution of Remaining Stiffness",
    xaxis_title="Remaining stiffness efficiency, θ",
    yaxis_title="Prior probability density",
    xaxis=dict(range=[0.01, 1.0]),
    template="plotly_white"
)

fig.show()

# 2. Structural Likelihood Formulation

The measurement model is  
$$Y_k = \theta K_{\text{nominal}} e^{\epsilon_k}, \quad \epsilon_k \sim \mathcal{N}(0, \sigma^2).$$

Taking logarithms,  
$$\ln Y_k = \ln(\theta K_{\text{nominal}}) + \epsilon_k.$$

Therefore,  
$$\ln Y_k \mid \Theta = \theta \sim \mathcal{N}\left(\ln(\theta K_{\text{nominal}}), \sigma^2\right).$$

Hence, $Y_k \mid \Theta = \theta$ follows a log-normal distribution.

# Likelihood of one sensor reading

The log-normal density is  
$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left[-\frac{(\ln y_k - \ln(\theta K_{\text{nominal}}))^2}{2\sigma^2}\right]$$

for  
$$y_k > 0.$$
Equivalently,

$$L(y_k | \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp \left[ -\frac{\ln \left( \frac{y_k}{\theta K_{\text{nominal}}} \right)^2}{2\sigma^2} \right]$$

---

# Joint likelihood of the running history

Let

$$y^{(k)} = (y_1, y_2, \dots, y_k).$$

Assuming the measurements are conditionally independent given $\Theta = \theta$,

$$L(y^{(k)} | \theta) = \prod_{i=1}^k L(y_i | \theta).$$

Therefore,

$$L(y^{(k)} | \theta) = \prod_{i=1}^k \frac{1}{y_i \sigma \sqrt{2\pi}} \exp \left[ -\frac{\ln \left( \frac{y_i}{\theta K_{\text{nominal}}} \right)^2}{2\sigma^2} \right].$$
This can also be written as

$$L(y^{(k)} | \theta) = \left[ \prod_{i=1}^k \frac{1}{y_i \sigma \sqrt{2\pi}} \right] \exp \left[ -\frac{1}{2\sigma^2} \sum_{i=1}^k \left[ \ln \left( \frac{y_i}{\theta K_{\text{nominal}}} \right) \right]^2 \right].$$

The product outside the exponential does not depend on $\theta$, so the likelihood up to proportionality is

$$L(y^{(k)} | \theta) \propto \exp \left[ -\frac{1}{2\sigma^2} \sum_{i=1}^k \left[ \ln \left( \frac{y_i}{\theta K_{\text{nominal}}} \right) \right]^2 \right].$$
# 3. Mathematical Formulation of the Non-Conjugate Grid Update

Suppose the posterior at step $k - 1$ is

$$f_{k-1}(\theta) = f_{\Theta|Y^{(k-1)}} \left( \theta \mid y^{(k-1)} \right).$$

When a new sensor reading $y_k$ is obtained, Bayes' theorem gives

$$f_k(\theta) = \frac{L(y_k \mid \theta) f_{k-1}(\theta)}{\int_{0.01}^{1} L(y_k \mid u) f_{k-1}(u) \, du}.$$

Up to a proportionality constant,

$$f_k(\theta) \propto L(y_k \mid \theta) f_{k-1}(\theta)$$

or

$$f_k(\theta) \propto \exp \left[ -\frac{\ln \left( \frac{y_k}{\theta K_{\text{nominal}}} \right)^2}{2\sigma^2} \right] f_{k-1}(\theta).$$

Starting from the initial prior, the complete posterior after $k$ readings is

$$f_k(\theta) \propto f_0(\theta) \prod_{i=1}^k L(y_i \mid \theta).$$
Therefore,

$$f_k(\theta) \propto \theta^7 (1 - \theta)^{0.5} \exp \left[ -\frac{1}{2\sigma^2} \sum_{i=1}^k \left[ \ln \left( \frac{y_i}{\theta K_{\text{nominal}}} \right) \right]^2 \right].$$

# Why this model is non-conjugate

A Beta density has the form

$$f(\theta) \propto \theta^{\alpha-1} (1 - \theta)^{\beta-1}.$$

However, the log-normal likelihood contains

$$\exp \left[ -\frac{(\ln y_k - \ln \theta - \ln K_{\text{nominal}})^2}{2\sigma^2} \right].$$

This introduces terms involving

$$(\ln \theta)^2$$

and $\ln \theta$, which cannot be combined with the Beta prior to produce another Beta distribution.

Therefore, unlike the Beta–Bernoulli model, there are no simple analytical parameter updates such as

$$\alpha_k = \alpha_{k-1} + y_k.$$
The posterior must be evaluated numerically using methods such as:

- fixed-grid integration;
- numerical quadrature;
- Laplace approximation;
- Markov chain Monte Carlo.

For this problem, a fixed bounded grid is used.
# 4. Running Point Estimates

After step $k$, let the normalized posterior be  
$$f_k(\theta) = f_{\Theta|Y^{(k)}}(\theta | Y^{(k)}).$$

## Running posterior mean

The Bayes estimate under squared-error loss is  
$$\hat{\theta}_{\text{Bayes}}^{(k)} = E[\Theta | Y^{(k)}] = \int_{0.01}^{1} \theta f_k(\theta) \, d\theta$$

provided that $f_k$ has already been normalized.

For an unnormalized posterior $\tilde{f}_k(\theta)$, the posterior mean is  
$$\hat{\theta}_{\text{Bayes}}^{(k)} = \frac{\int_{0.01}^{1} \theta \tilde{f}_k(\theta) \, d\theta}{\int_{0.01}^{1} \tilde{f}_k(\theta) \, d\theta}.$$

## Running MAP estimate

The MAP estimate is the value of $\theta$ where the posterior density is largest:  
$$\hat{\theta}_{\text{MAP}}^{(k)} = \arg \max_{0.01 \leq \theta \leq 1} f_k(\theta).$$
Unlike the posterior mean, the MAP estimate is not calculated using an integral. On a numerical grid it is approximated by

$$\hat{\theta}_{\text{MAP}}^{(k)} \approx \theta_{j^*}, \quad j^* = \arg \max_j f_k(\theta_j).$$

# Posterior variance

Posterior certainty may be measured using

$$\text{Var} \left( \Theta \mid \mathbf{y}^{(k)} \right) = \int_{0.01}^{1} \left( \theta - \hat{\theta}_{\text{Bayes}}^{(k)} \right)^2 f_k(\theta) \, d\theta.$$

As this variance decreases, the posterior density becomes narrower and the monitoring system becomes more confident.

In [8]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta


# =========================================================
# Functions
# =========================================================

def lognormal_likelihood(y, theta_grid, k_nominal, sigma):
    """
    Evaluate the log-normal likelihood of one sensor
    reading over the fixed theta grid.
    """
    log_ratio = np.log(
        y / (theta_grid * k_nominal)
    )

    likelihood = (
        1.0 / (y * sigma * np.sqrt(2.0 * np.pi))
    ) * np.exp(
        -(log_ratio ** 2) / (2.0 * sigma ** 2)
    )

    return likelihood


def calculate_posterior_mean(theta_grid, posterior):
    """
    Calculate the posterior mean using trapezoidal
    numerical integration.
    """
    return np.trapezoid(
        theta_grid * posterior,
        theta_grid
    )


def calculate_posterior_variance(
    theta_grid,
    posterior,
    posterior_mean
):
    """
    Calculate the posterior variance using trapezoidal
    numerical integration.
    """
    return np.trapezoid(
        (theta_grid - posterior_mean) ** 2 * posterior,
        theta_grid
    )


def calculate_credible_interval(
    theta_grid,
    posterior,
    probability=0.95
):
    """
    Calculate an equal-tailed credible interval from
    the numerical posterior distribution.
    """
    grid_widths = np.diff(theta_grid)

    cumulative_probability = np.concatenate(
        (
            [0.0],
            np.cumsum(
                0.5
                * (posterior[:-1] + posterior[1:])
                * grid_widths
            )
        )
    )

    cumulative_probability /= cumulative_probability[-1]

    lower_probability = (1.0 - probability) / 2.0
    upper_probability = 1.0 - lower_probability

    lower_limit = np.interp(
        lower_probability,
        cumulative_probability,
        theta_grid
    )

    upper_limit = np.interp(
        upper_probability,
        cumulative_probability,
        theta_grid
    )

    return lower_limit, upper_limit


# =========================================================
# Model parameters
# =========================================================

rng = np.random.default_rng(42)

theta_true = 0.68
k_nominal = 50.0
sigma = 0.15
n_measurements = 15

alpha_0 = 8.0
beta_0 = 1.5

# The lower limit is greater than zero because log(0)
# is undefined in the likelihood.
theta_grid = np.linspace(0.01, 1.0, 5000)


# =========================================================
# Task 1: Initial prior distribution
# =========================================================

prior_density = beta.pdf(
    theta_grid,
    a=alpha_0,
    b=beta_0
)

# Normalize the prior over the computational domain
prior_density /= np.trapezoid(
    prior_density,
    theta_grid
)

prior_mean = calculate_posterior_mean(
    theta_grid,
    prior_density
)

prior_map = theta_grid[
    np.argmax(prior_density)
]

print("Initial prior distribution: Beta(8, 1.5)")
print(f"Expected prior stiffness: {prior_mean:.4f}")
print(f"Prior MAP stiffness: {prior_map:.4f}")


# Plot the initial prior
prior_figure = go.Figure()

prior_figure.add_trace(
    go.Scatter(
        x=theta_grid,
        y=prior_density,
        mode="lines",
        name="Beta(8, 1.5) prior"
    )
)

prior_figure.update_layout(
    title="Initial Prior Distribution of Remaining Stiffness",
    xaxis_title="Remaining stiffness efficiency, θ",
    yaxis_title="Prior probability density",
    xaxis=dict(range=[0.01, 1.0]),
    template="plotly_white"
)

prior_figure.show()


# =========================================================
# Task 6: Sequential sensor simulation and updating
# =========================================================

posterior = prior_density.copy()

milestones = [0, 1, 2, 5, 10, 15]

posterior_curves = {
    0: posterior.copy()
}

measurements = []

posterior_means = [
    calculate_posterior_mean(
        theta_grid,
        posterior
    )
]

map_estimates = [
    theta_grid[np.argmax(posterior)]
]

initial_variance = calculate_posterior_variance(
    theta_grid,
    posterior,
    posterior_means[0]
)

posterior_standard_deviations = [
    np.sqrt(initial_variance)
]

credible_intervals = [
    calculate_credible_interval(
        theta_grid,
        posterior
    )
]


# Sequential Bayesian updating
for k in range(1, n_measurements + 1):

    # Generate random multiplicative noise
    epsilon_k = rng.normal(
        loc=0.0,
        scale=sigma
    )

    # Simulate the sensor reading
    y_k = (
        theta_true
        * k_nominal
        * np.exp(epsilon_k)
    )

    measurements.append(y_k)

    # Calculate the likelihood over the theta grid
    likelihood = lognormal_likelihood(
        y=y_k,
        theta_grid=theta_grid,
        k_nominal=k_nominal,
        sigma=sigma
    )

    # Calculate the unnormalized posterior
    unnormalized_posterior = (
        posterior * likelihood
    )

    # Normalize using the trapezoidal rule
    normalization_constant = np.trapezoid(
        unnormalized_posterior,
        theta_grid
    )

    if (
        not np.isfinite(normalization_constant)
        or normalization_constant <= 0
    ):
        raise ValueError(
            "Posterior normalization failed."
        )

    posterior = (
        unnormalized_posterior
        / normalization_constant
    )

    # Calculate the running posterior mean
    posterior_mean_k = calculate_posterior_mean(
        theta_grid,
        posterior
    )

    # Calculate the running MAP estimate
    map_estimate_k = theta_grid[
        np.argmax(posterior)
    ]

    # Calculate the running posterior variance
    posterior_variance_k = (
        calculate_posterior_variance(
            theta_grid,
            posterior,
            posterior_mean_k
        )
    )

    # Calculate the running credible interval
    credible_interval_k = (
        calculate_credible_interval(
            theta_grid,
            posterior
        )
    )

    posterior_means.append(
        posterior_mean_k
    )

    map_estimates.append(
        map_estimate_k
    )

    posterior_standard_deviations.append(
        np.sqrt(posterior_variance_k)
    )

    credible_intervals.append(
        credible_interval_k
    )

    if k in milestones:
        posterior_curves[k] = posterior.copy()


# =========================================================
# Determine sustained convergence
# =========================================================

tolerance = 0.03
sustained_convergence_step = None

for k in range(n_measurements + 1):

    remains_converged = all(
        abs(posterior_means[j] - theta_true) <= tolerance
        and abs(map_estimates[j] - theta_true) <= tolerance
        for j in range(k, n_measurements + 1)
    )

    if remains_converged:
        sustained_convergence_step = k
        break


# =========================================================
# Print numerical results
# =========================================================

print("\nSimulated sensor readings:")

for k, measurement in enumerate(
    measurements,
    start=1
):
    print(
        f"k = {k:2d}, "
        f"y_k = {measurement:.4f} kN/mm"
    )

print("\nEstimator values at selected milestones:")

for k in milestones:
    print(
        f"k = {k:2d}, "
        f"Posterior mean = {posterior_means[k]:.4f}, "
        f"MAP = {map_estimates[k]:.4f}"
    )

print(
    f"\nFinal posterior mean: "
    f"{posterior_means[-1]:.4f}"
)

print(
    f"Final MAP estimate: "
    f"{map_estimates[-1]:.4f}"
)

print(
    f"Final posterior standard deviation: "
    f"{posterior_standard_deviations[-1]:.4f}"
)

print(
    f"Final 95% credible interval: "
    f"({credible_intervals[-1][0]:.4f}, "
    f"{credible_intervals[-1][1]:.4f})"
)

print(
    f"True remaining stiffness: "
    f"{theta_true:.4f}"
)

print(
    f"First sustained convergence within ±{tolerance:.2f}: "
    f"k = {sustained_convergence_step}"
)


# =========================================================
# Plot 1: Posterior density curves
# =========================================================

posterior_figure = go.Figure()

for milestone in milestones:
    posterior_figure.add_trace(
        go.Scatter(
            x=theta_grid,
            y=posterior_curves[milestone],
            mode="lines",
            name=f"k = {milestone}"
        )
    )

posterior_figure.add_vline(
    x=theta_true,
    line_dash="dash",
    annotation_text="True θ = 0.68",
    annotation_position="top"
)

posterior_figure.update_layout(
    title=(
        "Sequential Posterior Density Curves "
        "for Remaining Stiffness"
    ),
    xaxis_title="Remaining stiffness efficiency, θ",
    yaxis_title="Posterior probability density",
    xaxis=dict(range=[0.01, 1.0]),
    template="plotly_white",
    hovermode="x unified"
)

posterior_figure.show()


# =========================================================
# Plot 2: Running posterior mean and MAP estimates
# =========================================================

steps = np.arange(n_measurements + 1)

estimator_figure = go.Figure()

estimator_figure.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode="lines+markers",
        name="Posterior Mean"
    )
)

estimator_figure.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP Estimate"
    )
)

estimator_figure.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True θ = 0.68",
    annotation_position="top right"
)

estimator_figure.update_layout(
    title=(
        "Convergence of Running Structural "
        "Stiffness Estimates"
    ),
    xaxis_title="Number of sensor measurements, k",
    yaxis_title="Estimated remaining stiffness",
    xaxis=dict(
        tickmode="linear",
        tick0=0,
        dtick=1
    ),
    yaxis=dict(range=[0.5, 1.0]),
    template="plotly_white",
    hovermode="x unified"
)

estimator_figure.show()

Initial prior distribution: Beta(8, 1.5)
Expected prior stiffness: 0.8421
Prior MAP stiffness: 0.9333



Simulated sensor readings:
k =  1, y_k = 35.5901 kN/mm
k =  2, y_k = 29.0891 kN/mm
k =  3, y_k = 38.0510 kN/mm
k =  4, y_k = 39.1518 kN/mm
k =  5, y_k = 25.3735 kN/mm
k =  6, y_k = 27.9672 kN/mm
k =  7, y_k = 34.6583 kN/mm
k =  8, y_k = 32.4248 kN/mm
k =  9, y_k = 33.9144 kN/mm
k = 10, y_k = 29.9163 kN/mm
k = 11, y_k = 38.7942 kN/mm
k = 12, y_k = 38.2074 kN/mm
k = 13, y_k = 34.3384 kN/mm
k = 14, y_k = 40.2636 kN/mm
k = 15, y_k = 36.4699 kN/mm

Estimator values at selected milestones:
k =  0, Posterior mean = 0.8421, MAP = 0.9333
k =  1, Posterior mean = 0.7947, MAP = 0.7972
k =  2, Posterior mean = 0.6977, MAP = 0.6877
k =  5, Posterior mean = 0.6823, MAP = 0.6780
k = 10, Posterior mean = 0.6577, MAP = 0.6554
k = 15, Posterior mean = 0.6873, MAP = 0.6857

Final posterior mean: 0.6873
Final MAP estimate: 0.6857
Final posterior standard deviation: 0.0266
Final 95% credible interval: (0.6367, 0.7408)
True remaining stiffness: 0.6800
First sustained convergence within ±0.03: k = 5


# Q. Gaussian Mixture Clustering as Conditional Updating

Consider a dataset
$$
x_1,x_2,\dots,x_n\in\mathbb R^d.
$$
We wish to cluster these observations into $K$ groups. Instead of assigning each point deterministically to a cluster at the beginning, we introduce a latent random variable
$$
C_i\in{1,\dots,K},
$$
where $C_i=k$ means that the observation $x_i$ belongs to cluster $k$.
Let the prior probability of cluster membership be
$$
P(C_i=k)=\phi_k,
$$
where
$$
\phi_k\ge 0,
\qquad
\sum_{k=1}^K \phi_k=1.
$$

Conditional on $C_i=k$, assume that the observation $X_i$ is generated from a multivariate Gaussian distribution:
$$
X_i\mid C_i=k
\sim
\mathscr N(\mu_k,\Sigma_k),
$$
where
$$
\mu_k\in\mathbb R^d,
\qquad
\Sigma_k\in\mathbb R^{d\times d}
$$
are the mean vector and covariance matrix of cluster $k$.

The model parameters
$$
\phi_k,\mu_k,\Sigma_k,
\qquad k=1,\dots,K,
$$
are assumed to be fixed but unknown.

---

1. Deriving the Marginal Density:
Using the law of total probability, show that the marginal density of $X_i$ is
$$
p(x_i)=\sum_{k=1}^K
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k).
$$
Explain why this density is called a Gaussian mixture density.

---

2. Deriving the Posterior Cluster Probability:
For a fixed observation $x_i$, use Bayes' rule to derive
$$
P(C_i=k\mid X_i=x_i)=\frac{
P(X_i=x_i\mid C_i=k)P(C_i=k)
}{
\sum_{j=1}^K P(X_i=x_i\mid C_i=j)P(C_i=j)
}.
$$
Then substitute the Gaussian model and the cluster prior to obtain
$$
P(C_i=k\mid X_i=x_i)=\frac{
\phi_k\mathscr N(x_i\mid \mu_k,\Sigma_k)
}{
\sum_{j=1}^K
\phi_j\mathscr N(x_i\mid \mu_j,\Sigma_j)
}.
$$
This quantity is called the responsibility of cluster $k$ for data point $x_i$, and is denoted by
$$
\gamma_{ik}=P(C_i=k\mid X_i=x_i).
$$
Explain why $\gamma_{ik}$ may be interpreted as a posterior probability of cluster membership.

---

3. One-Hot Encoding of the Latent Cluster Variable:
Now define a one-hot encoded latent random vector
$$
Z_i=
\begin{bmatrix}
Z_{i1}\\
Z_{i2}\\
\vdots\\
Z_{iK}
\end{bmatrix},
$$
where
$$
Z_{ik}=\begin{cases}
1, & \text{if } C_i=k,\\
0, & \text{otherwise}.
\end{cases}
$$
Show that
$$
\mathbb E[Z_{ik}\mid X_i=x_i]=P(C_i=k\mid X_i=x_i).
$$
Hence show that
$$
\mathbb E[Z_i\mid X_i=x_i]=\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}.
$$
Conclude that the soft cluster assignment in a Gaussian mixture model is precisely the conditional expectation
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

---

4. From Soft Assignment to Hard Clustering:
The vector
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
gives a soft assignment of $x_i$ to all clusters. A hard cluster assignment can be obtained by choosing the cluster with the largest posterior probability:
$$
\widehat C_i=\operatorname{arg\,max}_{1\le k\le K}
\gamma_{ik}.
$$
Explain the difference between soft clustering and hard clustering in this context.

---

5. Conditional Expectation of the Observation Given the Cluster:
Show that
$$
\mathbb E[X_i\mid C_i=k]=\mu_k.
$$
Explain why $\mu_k$ can be interpreted as the center of cluster $k$.
Then compare the two conditional expectations
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
and
$$
\mathbb E[X_i\mid C_i=k].
$$
Explain why the first gives the soft cluster membership of an observed point, while the second gives the mean location of a cluster.

---

6. The Complete-Data Likelihood
If the latent labels $z_i$ were known, the complete-data likelihood would be
$$
p(x_1,\dots,x_n,z_1,\dots,z_n)=\prod_{i=1}^n
\prod_{k=1}^K
\left[
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$
Take the logarithm and show that the complete-data log-likelihood is
$$
\ell_c=\sum_{i=1}^n
\sum_{k=1}^K
z_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why this expression would be easy to maximize if the values of $z_{ik}$ were known.

---

7. The EM Interpretation:
In practice, the latent variables $Z_i$ are not observed. The EM algorithm replaces the unknown indicators $z_{ik}$ by their conditional expectations given the observed data and current parameter estimates:
$$
z_{ik}
\quad\leadsto\quad
\mathbb E[Z_{ik}\mid X_i=x_i].
$$
That is,
$$
z_{ik}
\quad\leadsto\quad
\gamma_{ik}.
$$
This is the E-step of the EM algorithm.
Using this idea, write the expected complete-data log-likelihood:
$$
Q=\sum_{i=1}^n
\sum_{k=1}^K
\gamma_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why the E-step can be interpreted as a conditional update of cluster membership probabilities.

---

8. Parameter Updates:
By maximizing $Q$ with respect to the model parameters, derive the standard GMM updates:
$$
N_k=\sum_{i=1}^n \gamma_{ik},
$$
$$
\phi_k^{\text{new}}=\frac{N_k}{n},
$$
$$
\mu_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}x_i,
$$
and
$$
\Sigma_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}
(x_i-\mu_k^{\text{new}})
(x_i-\mu_k^{\text{new}})^T.
$$
Explain how the responsibility $\gamma_{ik}$ acts as a fractional membership weight of observation $x_i$ in cluster $k$.

---

9. Interpretation:
Write a short paragraph explaining why GMM clustering can be viewed as a repeated process of conditional updating.
Your answer should mention the following points:

* The mixture weight $\phi_k$ is the prior probability of cluster $k$.
* The Gaussian density $\mathscr N(x_i\mid \mu_k,\Sigma_k)$ measures how compatible $x_i$ is with cluster $k$.
* The responsibility $\gamma_{ik}$ is the posterior probability of cluster $k$ after observing $x_i$.
* The soft assignment vector is
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

* The M-step updates the cluster parameters using these posterior membership probabilities as weights.
Conclude that Gaussian mixture clustering is probabilistic clustering based on conditional expectations of latent cluster membership variables.

---

Here is a perfectly tailored question that you can add as the final part (**Part 10**) of your assignment notebook to bridge your theoretical derivations with your code implementation:

---

10. Computational Simulation and Out-of-Sample Validation

Using the theoretical framework established in the previous parts, write a Python class named `GMMFinancialSegmenter` that implements a two-dimensional Gaussian Mixture Model (GMM) using `scikit-learn` and visualizes the results interactively using `Plotly`. Your implementation should fulfill the following criteria:

* **Data Splitting and Scaling:** Accept a dataset containing two continuous features (e.g., mimicking financial behaviors like `PURCHASES` and `CREDIT_LIMIT`), standardize the features to handle variance scaling, and split the data into an 80% training set and a 20% validation/test set.
* **EM Execution:** Fit a GMM with $K=3$ components on the training data using the Expectation-Maximization (EM) algorithm, printing whether the model successfully converged and the number of iterations required.
* **Out-of-Sample Performance:** Compute and output the average log-likelihood score over the unseen test set to validate how well the learned density functions generalize to new data.
* **Interactive Visualizations:** Implement methods to generate three distinct Plotly figures:
1. An empirical **2D Density Heatmap** of the raw training data with marginal distributions to inspect its underlying multimodal structure.
2. A **Training Assignment Plot** that overlays the training data points on top of a continuous contour map showing the maximum posterior responsibilities ($\gamma_{ik}$) computed across a fine coordinate grid.
3. A **Test Assignment Plot** that replicates the contour boundary visualization but overlays out-of-sample test data points to expose the physical regions of cluster ambiguity.



Briefly evaluate the resulting plots. Explain how the continuous background contour map visually demonstrates the soft assignment expectation vector $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$ that you proved analytically in Part 3.

Use the dataset

https://www.kaggle.com/datasets/arjunbhasin2013/ccdata

# 1. Deriving the Marginal Density

Using the law of total probability,  
$$p(x_i) = \sum_{k=1}^K p(x_i, C_i = k).$$

Using the product rule,  
$$p(x_i, C_i = k) = p(x_i \mid C_i = k) P(C_i = k).$$

Therefore,  
$$p(x_i) = \sum_{k=1}^K p(x_i \mid C_i = k) P(C_i = k).$$

Since  
$$p(x_i \mid C_i = k) = \mathcal{N}(x_i \mid \mu_k, \Sigma_k)$$

and  
$$P(C_i = k) = \phi_k,$$

the marginal density is  
$$p(x_i) = \sum_{k=1}^K \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k).$$
where

$$\mathcal{N}(x_i | \mu_k, \Sigma_k) = \frac{1}{(2\pi)^{d/2} |\Sigma_k|^{1/2}} \exp \left[ -\frac{1}{2} (x_i - \mu_k)^T \Sigma_k^{-1} (x_i - \mu_k) \right].$$

This is called a **Gaussian mixture density** because the overall probability density is a weighted combination of $K$ Gaussian densities. Each Gaussian represents one cluster, while $\phi_k$ determines the contribution of cluster $k$ to the complete population.
## 2. Deriving the Posterior Cluster Probability

For an observed point $\mathbf{x}_i$, Bayes' rule gives

$$P(C_i = k \mid \mathbf{X}_i = \mathbf{x}_i) = \frac{P(\mathbf{X}_i = \mathbf{x}_i \mid C_i = k)P(C_i = k)}{P(\mathbf{X}_i = \mathbf{x}_i)}.$$

Using the marginal density from Task 1,

$$P(\mathbf{X}_i = \mathbf{x}_i) = \sum_{j=1}^K P(\mathbf{X}_i = \mathbf{x}_i \mid C_i = j)P(C_i = j).$$

Therefore,

$$P(C_i = k \mid \mathbf{X}_i = \mathbf{x}_i) = \frac{P(\mathbf{X}_i = \mathbf{x}_i \mid C_i = k)P(C_i = k)}{\sum_{j=1}^K P(\mathbf{X}_i = \mathbf{x}_i \mid C_i = j)P(C_i = j)}.$$

Substituting the Gaussian likelihood and mixture prior gives
$$P(C_i = k \mid X_i = x_i) = \frac{\phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^K \phi_j \mathcal{N}(x_i \mid \mu_j, \Sigma_j)}$$
This posterior probability is called the **responsibility** of cluster $k$ for observation $\mathbf{x}_i$:

$$\gamma_{ik} = P(C_i = k \mid \mathbf{X}_i = \mathbf{x}_i).$$
This posterior probability is called the **responsibility** of cluster $k$ for observation $x_i$:

$$\gamma_{ik} = P(C_i = k \mid X_i = x_i).$$

The value $\gamma_{ik}$ measures how strongly cluster $k$ is responsible for generating observation $x_i$. It satisfies

$$0 \leq \gamma_{ik} \leq 1$$

and

$$\sum_{k=1}^K \gamma_{ik} = 1.$$

Thus, $\gamma_{ik}$ is the posterior probability of cluster membership after observing $x_i$.
# 3. One-Hot Encoding of the Latent Cluster Variable

Define the one-hot encoded latent vector

$$\mathbf{Z}_i =
\begin{bmatrix}
Z_{i1} \\
Z_{i2} \\
\vdots \\
Z_{iK}
\end{bmatrix},$$

where

$$Z_{ik} =
\begin{cases}
1, & C_i = k, \\
0, & \text{otherwise}.
\end{cases}$$

Because $Z_{ik}$ is an indicator variable,

$$E[Z_{ik} \mid \mathbf{X}_i = \mathbf{x}_i] = 1 \cdot P(Z_{ik} = 1 \mid \mathbf{X}_i = \mathbf{x}_i) + 0 \cdot P(Z_{ik} = 0 \mid \mathbf{X}_i = \mathbf{x}_i).$$

Since $Z_{ik} = 1$ means $C_i = k$,

$$E[Z_{ik} \mid \mathbf{X}_i = \mathbf{x}_i] = P(C_i = k \mid \mathbf{X}_i = \mathbf{x}_i).$$

Therefore,

$$E[Z_{ik} \mid \mathbf{X}_i = \mathbf{x}_i] = \gamma_{ik}.$$
Consequently,

$$E[\mathbf{Z}_i \mid \mathbf{X}_i = \mathbf{x}_i] =
\begin{bmatrix}
\gamma_{i1} \\
\gamma_{i2} \\
\vdots \\
\gamma_{iK}
\end{bmatrix}.$$

Hence, the soft cluster-assignment vector is exactly the conditional expectation

$$E[\mathbf{Z}_i \mid \mathbf{X}_i = \mathbf{x}_i].$$

Each component gives the posterior probability that observation $\mathbf{x}_i$ belongs to the corresponding cluster.
# 4. From Soft Assignment to Hard Clustering

The vector

$$E[\mathbf{Z}_i \mid \mathbf{X}_i = \mathbf{x}_i] =
\begin{bmatrix}
\gamma_{i1} \\
\gamma_{i2} \\
\vdots \\
\gamma_{iK}
\end{bmatrix}$$

provides a soft assignment of observation $\mathbf{x}_i$ to all $K$ clusters.

A hard cluster assignment is obtained by selecting the cluster with the largest posterior responsibility:

$$\tilde{C}_i = \arg \max_{1 \leq k \leq K} \gamma_{ik}.$$

In **soft clustering**, every observation is assigned a probability of membership in every cluster. For example,

$$(\gamma_{i1}, \gamma_{i2}, \gamma_{i3}) = (0.10, 0.65, 0.25)$$

indicates that observation $i$ has a 65% posterior probability of belonging to cluster 2.

In **hard clustering**, the same observation is assigned only to cluster 2 because cluster 2 has the largest responsibility. Hard clustering discards the remaining probability information, while soft clustering retains information about uncertainty and overlapping clusters.
# 5. Conditional Expectation of the Observation Given the Cluster

Since

$$X_i \mid C_i = k \sim \mathcal{N}(\mu_k, \Sigma_k),$$

the conditional expectation of a multivariate Gaussian distribution is its mean vector. Therefore,

$$E[X_i \mid C_i = k] = \mu_k.$$

Thus, $\mu_k$ represents the expected location of observations belonging to cluster $k$. It can therefore be interpreted as the center of cluster $k$.

The two conditional expectations have different meanings:

$$E[\mathbf{Z}_i \mid \mathbf{X}_i = \mathbf{x}_i] =
\begin{bmatrix}
\gamma_{i1} \\
\vdots \\
\gamma_{iK}
\end{bmatrix}$$

gives the posterior cluster-membership probabilities of an already observed point $\mathbf{x}_i$.

In contrast,

$$E[X_i \mid C_i = k] = \mu_k$$

gives the expected position of an observation generated from cluster $k$.

Therefore, the first expectation describes which cluster an observed point probably belongs to, while the second describes where observations from a particular cluster are expected to be located.
# 6. Complete-Data Likelihood

Suppose the latent one-hot cluster labels $z_1, \dots, z_n$ were observed.

For one observation,

$$p(x_i, z_i) = \prod_{k=1}^K \left[ \phi_k \mathcal{N}(x_i | \mu_k, \Sigma_k) \right]^{z_{ik}}.$$

Assuming observations are independent, the complete-data likelihood is

$$p(x_1, \dots, x_n, z_1, \dots, z_n) = \prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathcal{N}(x_i | \mu_k, \Sigma_k) \right]^{z_{ik}}.$$

Taking the logarithm,

$$\ell_c = \log p(x_1, \dots, x_n, z_1, \dots, z_n).$$

Using the logarithm of a product,

$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \log \left[ \phi_k \mathcal{N}(x_i | \mu_k, \Sigma_k) \right].$$
Therefore,

$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \log \phi_k + \log \mathcal{N}(x_i | \mu_k, \Sigma_k) \right].$$

If $z_{ik}$ were known, each observation would belong to one known cluster. The likelihood would then separate into $K$ independent weighted Gaussian estimation problems. There would be no logarithm of a sum of mixture densities, making the maximum-likelihood estimation of $\phi_k$, $\mu_k$ and $\Sigma_k$ straightforward.
# 7. EM Interpretation

In practice, the latent variables $Z_{ik}$ are not observed. The Expectation-Maximization algorithm replaces each unknown indicator by its conditional expectation using the current parameter estimates:

$$Z_{ik} \to E[Z_{ik} \mid X_i = x_i].$$

Therefore,

$$Z_{ik} \to \gamma_{ik}.$$

The expected complete-data log-likelihood is

$$Q = E[\ell_c \mid X].$$

Substituting the posterior responsibilities gives

$$Q = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \left[ \log \phi_k + \log \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right].$$

This is the **E-step** of the EM algorithm. At each iteration, the current mixture weights, means and covariance matrices act as the current model. Bayes' rule is then used to update the probability that each observation belongs to each cluster.

Therefore, the E-step is a conditional updating step:

prior cluster probability + data likelihood → posterior cluster probability.
# 8. Parameter Updates

Define the effective number of observations assigned to cluster $k$ as

$$N_k = \sum_{i=1}^n \gamma_{ik}.$$

## Mixture-weight update

The part of $Q$ involving the mixture weights is

$$\sum_{k=1}^K N_k \log \phi_k.$$

This is maximized subject to

$$\sum_{k=1}^K \phi_k = 1.$$

Using a Lagrange multiplier gives

$$\phi_k^{\text{new}} = \frac{N_k}{n}.$$
# Mean-vector update

The terms involving $\mu_k$ are maximized by setting the derivative of $Q$ with respect to $\mu_k$ equal to zero. This gives

$$\sum_{i=1}^n \gamma_{ik}(x_i - \mu_k^{\text{new}}) = 0.$$

Therefore,

$$\mu_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} x_i.$$

# Covariance-matrix update

Maximizing $Q$ with respect to $\Sigma_k$ gives

$$\Sigma_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} (x_i - \mu_k^{\text{new}})(x_i - \mu_k^{\text{new}})^T.$$

The responsibility $\gamma_{ik}$ acts as a fractional membership weight. For example, if

$$\gamma_{ik} = 0.80,$$

observation $\mathbf{x}_i$ contributes 80% of a complete observation to cluster $k$. If

$$\gamma_{ik} = 0.05,$$

it contributes only 5%. Therefore, the updated mean and covariance are weighted estimates in which observations with higher responsibilities have greater influence.

In [10]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.impute import SimpleImputer
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


class GMMFinancialSegmenter:
    """
    Two-dimensional Gaussian Mixture Model for financial
    customer segmentation using PURCHASES and CREDIT_LIMIT.
    """

    def __init__(
        self,
        n_components=3,
        test_size=0.20,
        random_state=42,
        n_init=10,
        max_iter=500
    ):
        self.n_components = n_components
        self.test_size = test_size
        self.random_state = random_state
        self.n_init = n_init
        self.max_iter = max_iter

        self.features = [
            "PURCHASES",
            "CREDIT_LIMIT"
        ]

        self.is_fitted = False

    def _load_data(self, data):
        """
        Accept either a Pandas DataFrame or a CSV file path.
        """
        if isinstance(data, pd.DataFrame):
            dataframe = data.copy()

        elif isinstance(data, (str, os.PathLike)):
            dataframe = pd.read_csv(data)

        else:
            raise TypeError(
                "data must be a Pandas DataFrame or a CSV file path."
            )

        missing_columns = [
            column
            for column in self.features
            if column not in dataframe.columns
        ]

        if missing_columns:
            raise ValueError(
                f"Missing required columns: {missing_columns}"
            )

        return dataframe

    def fit(self, data):
        """
        Prepare the data, fit the GMM and calculate
        training and test responsibilities.
        """
        dataframe = self._load_data(data)

        # Select the two required continuous variables
        X = dataframe[self.features].copy()

        # Perform the 80%/20% split before fitting the
        # imputer and scaler to prevent information leakage
        (
            X_train_raw,
            X_test_raw
        ) = train_test_split(
            X,
            test_size=self.test_size,
            random_state=self.random_state
        )

        self.X_train_raw_ = X_train_raw
        self.X_test_raw_ = X_test_raw

        # Replace missing values using medians calculated
        # only from the training data
        self.imputer_ = SimpleImputer(
            strategy="median"
        )

        X_train_imputed = self.imputer_.fit_transform(
            X_train_raw
        )

        X_test_imputed = self.imputer_.transform(
            X_test_raw
        )

        self.X_train_ = pd.DataFrame(
            X_train_imputed,
            columns=self.features,
            index=X_train_raw.index
        )

        self.X_test_ = pd.DataFrame(
            X_test_imputed,
            columns=self.features,
            index=X_test_raw.index
        )

        # Standardize the two variables using the
        # training-set means and standard deviations
        self.scaler_ = StandardScaler()

        self.X_train_scaled_ = self.scaler_.fit_transform(
            self.X_train_
        )

        self.X_test_scaled_ = self.scaler_.transform(
            self.X_test_
        )

        # Fit a three-component GMM using EM
        self.gmm_ = GaussianMixture(
            n_components=self.n_components,
            covariance_type="full",
            random_state=self.random_state,
            n_init=self.n_init,
            max_iter=self.max_iter,
            reg_covar=1e-6
        )

        self.gmm_.fit(
            self.X_train_scaled_
        )

        # Posterior responsibilities
        self.train_responsibilities_ = (
            self.gmm_.predict_proba(
                self.X_train_scaled_
            )
        )

        self.test_responsibilities_ = (
            self.gmm_.predict_proba(
                self.X_test_scaled_
            )
        )

        # Hard assignments obtained from maximum responsibility
        self.train_labels_ = np.argmax(
            self.train_responsibilities_,
            axis=1
        )

        self.test_labels_ = np.argmax(
            self.test_responsibilities_,
            axis=1
        )

        # Average log-likelihood on unseen data
        self.test_average_log_likelihood_ = (
            self.gmm_.score(
                self.X_test_scaled_
            )
        )

        # Convert GMM means from standardized units
        # back to the original financial units
        original_cluster_means = (
            self.scaler_.inverse_transform(
                self.gmm_.means_
            )
        )

        training_counts = np.bincount(
            self.train_labels_,
            minlength=self.n_components
        )

        self.cluster_summary_ = pd.DataFrame(
            {
                "Cluster": np.arange(
                    self.n_components
                ),
                "Mixture weight": self.gmm_.weights_,
                "Mean PURCHASES":
                    original_cluster_means[:, 0],
                "Mean CREDIT_LIMIT":
                    original_cluster_means[:, 1],
                "Training count": training_counts
            }
        )

        self.is_fitted = True

        print(
            f"Model converged: "
            f"{self.gmm_.converged_}"
        )

        print(
            f"Number of EM iterations: "
            f"{self.gmm_.n_iter_}"
        )

        print(
            f"Average test log-likelihood: "
            f"{self.test_average_log_likelihood_:.4f}"
        )

        print("\nCluster summary:")
        print(
            self.cluster_summary_.to_string(
                index=False
            )
        )

        return self

    def _check_fitted(self):
        if not self.is_fitted:
            raise RuntimeError(
                "The model must be fitted before creating plots."
            )

    def plot_empirical_density(self):
        """
        Plot 1:
        Empirical two-dimensional density heatmap of
        the raw training data with marginal histograms.
        """
        self._check_fitted()

        figure = px.density_heatmap(
            self.X_train_,
            x="PURCHASES",
            y="CREDIT_LIMIT",
            nbinsx=60,
            nbinsy=60,
            marginal_x="histogram",
            marginal_y="histogram",
            title=(
                "Empirical 2D Density of the "
                "Training Data"
            )
        )

        figure.update_layout(
            template="plotly_white",
            xaxis_title="PURCHASES",
            yaxis_title="CREDIT_LIMIT"
        )

        return figure

    def _create_responsibility_grid(
        self,
        grid_size=220
    ):
        """
        Create a fine grid in the original feature units
        and compute the GMM responsibilities on it.
        """
        self._check_fitted()

        purchases = self.X_train_[
            "PURCHASES"
        ].to_numpy()

        credit_limits = self.X_train_[
            "CREDIT_LIMIT"
        ].to_numpy()

        purchases_range = (
            purchases.max() - purchases.min()
        )

        credit_range = (
            credit_limits.max() - credit_limits.min()
        )

        purchases_padding = 0.04 * max(
            purchases_range,
            1.0
        )

        credit_padding = 0.04 * max(
            credit_range,
            1.0
        )

        purchases_grid = np.linspace(
            max(
                0.0,
                purchases.min() - purchases_padding
            ),
            purchases.max() + purchases_padding,
            grid_size
        )

        credit_grid = np.linspace(
            max(
                0.0,
                credit_limits.min() - credit_padding
            ),
            credit_limits.max() + credit_padding,
            grid_size
        )

        grid_x, grid_y = np.meshgrid(
            purchases_grid,
            credit_grid
        )

        raw_grid = pd.DataFrame(
            {
                "PURCHASES": grid_x.ravel(),
                "CREDIT_LIMIT": grid_y.ravel()
            }
        )

        scaled_grid = self.scaler_.transform(
            raw_grid
        )

        grid_responsibilities = (
            self.gmm_.predict_proba(
                scaled_grid
            )
        )

        maximum_responsibility = (
            grid_responsibilities.max(axis=1)
            .reshape(grid_x.shape)
        )

        winning_cluster = (
            grid_responsibilities.argmax(axis=1)
            .reshape(grid_x.shape)
        )

        return (
            purchases_grid,
            credit_grid,
            maximum_responsibility,
            winning_cluster
        )

    def _plot_assignments(
        self,
        split="train",
        grid_size=220
    ):
        """
        Create a contour map of maximum posterior
        responsibility and overlay either training
        or test observations.
        """
        self._check_fitted()

        if split == "train":
            plot_data = self.X_train_
            labels = self.train_labels_
            responsibilities = (
                self.train_responsibilities_
            )

            title = (
                "Training Assignments and Maximum "
                "Posterior Responsibilities"
            )

        elif split == "test":
            plot_data = self.X_test_
            labels = self.test_labels_
            responsibilities = (
                self.test_responsibilities_
            )

            title = (
                "Test Assignments and Maximum "
                "Posterior Responsibilities"
            )

        else:
            raise ValueError(
                "split must be either 'train' or 'test'."
            )

        (
            purchases_grid,
            credit_grid,
            maximum_responsibility,
            winning_cluster
        ) = self._create_responsibility_grid(
            grid_size=grid_size
        )

        figure = go.Figure()

        # Continuous contour map of max_k gamma_ik
        figure.add_trace(
            go.Contour(
                x=purchases_grid,
                y=credit_grid,
                z=maximum_responsibility,
                colorscale="Viridis",
                contours=dict(
                    start=1.0 / self.n_components,
                    end=1.0,
                    size=0.05
                ),
                colorbar=dict(
                    title="Maximum γ"
                ),
                opacity=0.72,
                name="Maximum responsibility",
                hovertemplate=(
                    "PURCHASES=%{x:.2f}<br>"
                    "CREDIT_LIMIT=%{y:.2f}<br>"
                    "Maximum γ=%{z:.3f}"
                    "<extra></extra>"
                )
            )
        )

        # Add lines separating the winning-cluster regions
        figure.add_trace(
            go.Contour(
                x=purchases_grid,
                y=credit_grid,
                z=winning_cluster,
                showscale=False,
                contours=dict(
                    coloring="none",
                    showlabels=False,
                    start=0.5,
                    end=self.n_components - 1.5,
                    size=1
                ),
                line=dict(
                    color="black",
                    width=2
                ),
                hoverinfo="skip",
                name="Cluster boundaries"
            )
        )

        # Overlay observations using hard assignments
        for cluster in range(
            self.n_components
        ):
            mask = labels == cluster

            figure.add_trace(
                go.Scatter(
                    x=plot_data.loc[
                        mask,
                        "PURCHASES"
                    ],
                    y=plot_data.loc[
                        mask,
                        "CREDIT_LIMIT"
                    ],
                    mode="markers",
                    name=f"Cluster {cluster}",
                    marker=dict(
                        size=5,
                        opacity=0.55
                    ),
                    customdata=responsibilities[
                        mask,
                        cluster
                    ],
                    hovertemplate=(
                        f"Cluster {cluster}<br>"
                        "PURCHASES=%{x:.2f}<br>"
                        "CREDIT_LIMIT=%{y:.2f}<br>"
                        "Assigned-cluster γ="
                        "%{customdata:.3f}"
                        "<extra></extra>"
                    )
                )
            )

        figure.update_layout(
            title=title,
            xaxis_title="PURCHASES",
            yaxis_title="CREDIT_LIMIT",
            template="plotly_white"
        )

        return figure

    def plot_training_assignments(
        self,
        grid_size=220
    ):
        """
        Plot 2:
        Training observations over the responsibility map.
        """
        return self._plot_assignments(
            split="train",
            grid_size=grid_size
        )

    def plot_test_assignments(
        self,
        grid_size=220
    ):
        """
        Plot 3:
        Test observations over the responsibility map.
        """
        return self._plot_assignments(
            split="test",
            grid_size=grid_size
        )


# =========================================================
# Run the model using the supplied dataset
# =========================================================

segmenter = GMMFinancialSegmenter(
    n_components=3,
    test_size=0.20,
    random_state=42
)

segmenter.fit("CC GENERAL.csv")


# Plot 1: Empirical density heatmap
density_figure = (
    segmenter.plot_empirical_density()
)
density_figure.show()


# Plot 2: Training assignments
training_figure = (
    segmenter.plot_training_assignments()
)
training_figure.show()


# Plot 3: Out-of-sample test assignments
test_figure = (
    segmenter.plot_test_assignments()
)
test_figure.show()

Model converged: True
Number of EM iterations: 19
Average test log-likelihood: -1.5941

Cluster summary:
 Cluster  Mixture weight  Mean PURCHASES  Mean CREDIT_LIMIT  Training count
       0        0.444880      188.881562        2092.528699            3283
       1        0.454698      928.916543        5801.914982            3251
       2        0.100422     4839.529642        9571.026588             626
